### In this notebook we show how the model to model evaluation module works

In [31]:
# STENCIL DEFINITIONS
# ==============================================================================

# Updated based on your complete stencil list
START_EVENT_STENCILS = [
    "StartNoneEvent",
    "StartMessageEvent",
    "StartTimerEvent",
    "StartSignalEvent",
    "StartConditionalEvent",
    "StartErrorEvent",
    "StartEscalationEvent",
    "StartCompensationEvent",
    "StartMultipleEvent",
    "StartParallelMultipleEvent",
]

END_EVENT_STENCILS = [
    "EndNoneEvent",
    "EndMessageEvent",
    "EndTerminateEvent",
    "EndErrorEvent",
    "EndEscalationEvent",
    "EndCompensationEvent",
    "EndCancelEvent",
    "EndMultipleEvent",
    "EndSignalEvent",
]

INTERMEDIATE_EVENT_STENCILS = [
    "IntermediateEvent",
    "IntermediateMessageEventCatching",
    "IntermediateMessageEventThrowing",
    "IntermediateTimerEvent",
    "IntermediateErrorEvent",
    "IntermediateConditionalEvent",
    "IntermediateEscalationEvent",
    "IntermediateEscalationEventThrowing",
    "IntermediateSignalEventThrowing",
    "IntermediateSignalEventCatching",
    "IntermediateCompensationEventCatching",
    "IntermediateCompensationEventThrowing",
    "IntermediateCancelEvent",
    "IntermediateMultipleEventCatching",
    "IntermediateMultipleEventThrowing",
    "IntermediateParallelMultipleEventCatching",
    "IntermediateLinkEventThrowing",
    "IntermediateLinkEventCatching",
]

GATEWAY_STENCILS = [
    "Exclusive_Databased_Gateway",
    "ParallelGateway",
    "InclusiveGateway",
    "ComplexGateway",
    "EventbasedGateway",
]

ACTIVITY_STENCILS = ["Task", "Subprocess", "CollapsedSubprocess", "EventSubprocess", "CollapsedEventSubprocess"]

# All flow nodes (elements that can have sequence flows)
FLOW_NODE_STENCILS = (
    START_EVENT_STENCILS + END_EVENT_STENCILS + INTERMEDIATE_EVENT_STENCILS + GATEWAY_STENCILS + ACTIVITY_STENCILS
)

# Supporting elements
SUPPORTING_STENCILS = [
    "DataObject",
    "DataStore",
    "TextAnnotation",
    "Group",
    "Message",
    "Association_Unidirectional",
    "Association_Undirected",
    "Association_Bidirectional",
    "MessageFlow",
    "ITSystem",
]

from collections import Counter


def all_stencils(bpmn_models):
    all_stencils = []
    for i in final_count(bpmn_models):
        if i[0] not in all_stencils:
            all_stencils.append(i[0])
    return all_stencils


def count_stencil_ids(obj):
    """Recursively counts occurrences of all stencil IDs in a Signavio model tree."""
    counts = Counter()

    if isinstance(obj, dict):
        # If this object has a 'stencil' with an 'id', count it
        stencil = obj.get("stencil")
        if isinstance(stencil, dict) and "id" in stencil:
            counts[stencil["id"]] += 1
        # Recurse into any childShapes
        childshapes = obj.get("childShapes")
        if isinstance(childshapes, list):
            for child in childshapes:
                counts.update(count_stencil_ids(child))
        # Optionally, you can search other dict values in case the model is nonstandard
        # for v in obj.values():
        #    counts.update(count_stencil_ids(v))

    elif isinstance(obj, list):
        for item in obj:
            counts.update(count_stencil_ids(item))

    return counts


def final_count(bpmn_models):
    """
    Counts all stencil IDs across multiple BPMN models and returns a sorted list of counts.
    """

    final_dict = {}
    for i in bpmn_models:
        result = dict(count_stencil_ids(i))
        for m in result.keys():
            if m in final_dict:
                final_dict[m] += 1
            else:
                final_dict[m] = 1
    return sorted(final_dict.items(), key=lambda x: x[1], reverse=True)


def extract_elements_by_stencil_ids(model, stencil_ids):
    """
    Recursively extract all elements matching any of the given stencil IDs.

    Args:
        model: The BPMN model (JSON structure)
        stencil_ids: List of stencil IDs to match

    Returns:
        List of matching elements
    """
    results = []
    shapes = model.get("childShapes", [])
    for shape in shapes:
        if shape.get("stencil", {}).get("id") in stencil_ids:
            results.append(shape)
        if "childShapes" in shape:
            results.extend(extract_elements_by_stencil_ids(shape, stencil_ids))
    return results

In [32]:
import sys, json

sys.path.append("../")
sys.path.append("../model_evaluation/")

In [33]:
# load some examples from examples folder in Signavio json format
filename_ground_truth = f"../examples/E_j04.json"
with open(filename_ground_truth, "r") as infile:
    E4 = json.load(infile)


filename_generated = f"../examples/E_j04_4.bpmn2 _ Signavio.json"
with open(filename_generated, "r") as infile:
    E4_1 = json.load(infile)


filename_generated = f"../examples/process_complex.json"
with open(filename_generated, "r") as infile:
    pc = json.load(infile)

filename_generated = f"../examples/misc_booking_flight_tickets.json"
with open(filename_generated, "r") as infile:
    misc_loan_ft = json.load(infile)

filename_generated = f"../examples/misc_credit_quote_creation.json"
with open(filename_generated, "r") as infile:
    misc_loan_credit = json.load(infile)

filename_generated = f"../examples/Adrians_ex.json"
with open(filename_generated, "r") as infile:
    adrians_ex = json.load(infile)

In [34]:
count_stencil_ids(misc_loan_ft)

Counter({'SequenceFlow': 16,
         'MessageFlow': 7,
         'IntermediateMessageEventCatching': 4,
         'Task': 3,
         'Pool': 2,
         'Lane': 2,
         'IntermediateMessageEventThrowing': 2,
         'ParallelGateway': 2,
         'BPMNDiagram': 1,
         'CollapsedPool': 1,
         'StartNoneEvent': 1,
         'EndNoneEvent': 1,
         'StartMessageEvent': 1,
         'DataStore': 1,
         'EndMessageEvent': 1,
         'Subprocess': 1,
         'IntermediateTimerEvent': 1,
         'Exclusive_Databased_Gateway': 1,
         'Association_Unidirectional': 1})

In [35]:
from BPMN_conversion import BPMNConverter
from bpmn_schema_helper import BPMNConverter as original_BPMNConverter

In [36]:
import json

# E4_json = BPMNConverter.convert(E4).to_json()

# E4_1_json = BPMNConverter.convert(E4_1).to_json()

misc_bft_json = BPMNConverter.convert(misc_loan_ft).to_json()
misc_bft_json_original = original_BPMNConverter.convert(E4_1).to_json()

misc_credit_json = BPMNConverter.convert(misc_loan_credit).to_json()

pc_json = BPMNConverter.convert(pc).to_json()
pc_original_json = original_BPMNConverter.convert(pc).to_json()

# adrians_json = BPMNConverter.convert(adrians_ex).to_json()
# write E4_json to file
# with open("../E4_minimal.json", "w") as outfile:
#     E4_1_json = BPMNConverter.convert(misc_ft)
#     json.dump(json.loads(E4_1_json.to_json()), outfile, indent=4)

In [37]:
json.loads(misc_bft_json), json.loads(misc_credit_json)

({'activities': [{'id': 'sid-E5689480-97F3-457D-B623-EF8AA5626B99',
    'name': 'Plan travels',
    'type': 'Task'},
   {'id': 'sid-C30095C3-EE47-46D1-992F-3D1CBAEE726E',
    'name': 'Select the best offer and request tickets',
    'type': 'Send'},
   {'id': 'sid-BDBF2D9E-F414-4EB1-99E0-1E9C4D319DFF',
    'name': 'Create schedule',
    'type': 'Task'},
   {'id': 'sid-BD21A3DC-0E6D-46A2-A745-5F69C3991B8E',
    'name': '',
    'type': 'Subprocess',
    'elemRefs': ['sid-044B6E47-536D-4262-B6A0-102D3C20A06A',
     'sid-619F6299-172A-4AFD-8C60-15267AF1D0C8']}],
  'events': [{'id': 'sid-9D6D682F-6E36-4DDD-BED8-15A5A5EB3296',
    'name': 'Feeling the Wanderlust',
    'type': 'StartNoneEvent'},
   {'id': 'sid-6DE31EA7-8E36-459E-80CC-4E9B21E358B9',
    'name': 'Send travel request',
    'type': 'IntermediateMessageEventThrowing'},
   {'id': 'sid-C3683C28-0D9F-4D82-A8BA-EC504988B902',
    'name': 'Get schedule',
    'type': 'IntermediateMessageEventCatching'},
   {'id': 'sid-08598060-42EA-4673-

In [38]:
def extract_atomic_names(bpmn, atomic_type):
    """
    atomic_type can be:
      'activity_names', 'activity_types', 'event_names', 'event_types',
      'gateway_names', 'gateway_types', 'pool_names', 'lane_names'
    """
    if atomic_type == "activity_names":
        # Includes tasks, all named subprocesses, etc.
        return set(a["name"] for a in bpmn.get("activities", []) if a.get("name"))
    elif atomic_type == "activity_types":
        return set(a["type"] for a in bpmn.get("activities", []) if a.get("type"))
    elif atomic_type == "event_names":
        return set(e["name"] for e in bpmn.get("events", []) if e.get("name"))
    elif atomic_type == "event_types":
        return set(e["type"] for e in bpmn.get("events", []) if e.get("type"))
    elif atomic_type == "gateway_names":
        return set(g["name"] for g in bpmn.get("gateways", []) if g.get("name"))
    elif atomic_type == "gateway_types":
        return set(g["type"] for g in bpmn.get("gateways", []) if g.get("type"))
    elif atomic_type == "pool_names":
        return set(p["name"] for p in bpmn.get("pools", []) if p.get("name"))
    elif atomic_type == "lane_names":
        names = set()
        for pool in bpmn.get("pools", []):
            for lane in pool.get("lanes", []):
                lname = lane.get("name", "")
                if lname:
                    names.add(lname)
        return names
    return set()


def build_name_mapping(names1, names2, similarity_func, threshold=0.7):
    """
    Returns a dict: {name2 value: name1 value (most similar) if similarity >= threshold}
    """
    mapping = {}
    compared_pairs = []
    for n2 in names2:
        best_score = -float("inf")
        best_n1 = None
        for n1 in names1:
            score = similarity_func(n1, n2)
            compared_pairs.append((n1, n2, score))
            if score > best_score:
                best_score = score
                best_n1 = n1
        if best_score >= threshold:
            mapping[n2] = best_n1
    return mapping


import copy


def apply_atomic_name_mapping(bpmn, mappings):
    """
    Returns a deepcopy of bpmn with atomic names in mappings replaced per mapping.
    mappings: {'activity_names': {old2: mapped1}, ...}
    """
    bpmn2 = copy.deepcopy(bpmn)
    # Activities name
    if "activity_names" in mappings:
        for activity in bpmn2.get("activities", []):
            name = activity.get("name", "")
            if name in mappings["activity_names"]:
                activity["name"] = mappings["activity_names"][name]
    # Activities type
    if "activity_types" in mappings:
        for activity in bpmn2.get("activities", []):
            typename = activity.get("type", "")
            if typename in mappings["activity_types"]:
                activity["type"] = mappings["activity_types"][typename]

    # Events
    if "event_names" in mappings:
        for event in bpmn2.get("events", []):
            name = event.get("name", "")
            if name in mappings["event_names"]:
                event["name"] = mappings["event_names"][name]

    if "event_types" in mappings:
        for activity in bpmn2.get("events", []):
            typename = activity.get("type", "")
            if typename in mappings["event_types"]:
                activity["type"] = mappings["event_types"][typename]

    # Gateways (using type, not name!)
    if "gateway_names" in mappings:
        for gateway in bpmn2.get("gateways", []):
            name = gateway.get("name", "")
            if name in mappings["gateway_names"]:
                gateway["name"] = mappings["gateway_names"][name]

    # Gateways (using type, not name!)
    if "gateway_types" in mappings:
        for gateway in bpmn2.get("gateways", []):
            typename = gateway.get("type", "")
            if typename in mappings["gateway_types"]:
                gateway["type"] = mappings["gateway_types"][typename]
    # Pools
    if "pool_names" in mappings:
        for pool in bpmn2.get("pools", []):
            name = pool.get("name", "")
            if name in mappings["pool_names"]:
                pool["name"] = mappings["pool_names"][name]
    # Lanes
    if "lane_names" in mappings:
        for pool in bpmn2.get("pools", []):
            for lane in pool.get("lanes", []):
                lname = lane.get("name", "")
                if lname in mappings["lane_names"]:
                    lane["name"] = mappings["lane_names"][lname]
    return bpmn2


def create_all_atomic_mappings(bpmn1, bpmn2, similarity_func, threshold=0.7):
    mapping_types = [
        "activity_names",
        "activity_types",
        "event_names",
        "event_types",
        "gateway_names",
        "gateway_types",
        "pool_names",
        "lane_names",
    ]
    mappings = {}
    for mt in mapping_types:
        names1 = extract_atomic_names(bpmn1, mt)
        names2 = extract_atomic_names(bpmn2, mt)
        this_mapping = build_name_mapping(names1, names2, similarity_func, threshold)
        if this_mapping:
            mappings[mt] = this_mapping
    return mappings


def normalize_atomic_names(model1, model2, similarity_func, threshold=0.7):
    mappings = create_all_atomic_mappings(model1, model2, similarity_func, threshold)
    model2_aligned = apply_atomic_name_mapping(model2, mappings)
    return model2_aligned, mappings


from model_evaluation.string_similarity import bert_cosine_optimized


def similarity_SFA(list1, list2, method="dice", frequency_aware=False):
    """Similarity metric for lists (optionally frequency/multiset aware)."""
    if frequency_aware:
        list1, list2 = index_list(list1), index_list(list2)
    else:
        list1, list2 = list(set(list1)), list(set(list2))
    if method == "dice":
        return dice_list(list1, list2)
    elif method == "jaccard":
        return jaccard_list(list1, list2)
    elif method in {"precision", "recall", "f1"}:
        return scores(list1, list2, score_type=method)


# # 1. Normalize Model2 atomic names:
# model2_aligned, mappings = normalize_atomic_names(model1, model2, bert_cosine_optimized, threshold=0.7)

# # 2. Extract sets from both:
# sets1 = extract_bpmn_sets(model1)
# sets2 = extract_bpmn_sets(model2_aligned)

# # 3. Compare
# score = similarity_SFA(sets1['activity_names'], sets2['activity_names'])

In [44]:
# extract_atomic_names(json.loads(misc_bft_json), "gateway_names")

create_all_atomic_mappings(json.loads(misc_bft_json), new_json, bert_cosine_optimized, threshold=0.2)

{'activity_names': {'create schedule': 'Create schedule',
  'select best offer & request ticket': 'Select the best offer and request tickets',
  'plan travel': 'Plan travels'},
 'activity_types': {'Send': 'Send',
  'Subprocess': 'Subprocess',
  'Task': 'Task'},
 'event_names': {'two min': '2 min',
  'Wanderlust': 'Feeling the Wanderlust',
  'Confirm received': 'Receive confirmation',
  'request arrived': 'Customer request processed',
  'Receive e-ticket': 'Receive eTickets',
  'Request processed': 'Customer request processed',
  'send travel req': 'Send travel request',
  'Trip can start': 'Travel can begin',
  'schedule received': 'Get schedule'},
 'event_types': {'IntermediateTimerEvent': 'IntermediateTimerEvent',
  'StartNoneEvent': 'StartNoneEvent',
  'IntermediateMessageEventThrowing': 'IntermediateMessageEventThrowing',
  'IntermediateMessageEventCatching': 'IntermediateMessageEventCatching',
  'EndNoneEvent': 'EndNoneEvent',
  'EndMessageEvent': 'EndMessageEvent',
  'StartMessag

In [40]:
# model2_aligned, mappings = normalize_atomic_names(
#     json.loads(misc_bft_json), json.loads(misc_credit_json), bert_cosine_optimized, threshold=0.8
# )
# # # 2. Extract sets from both:
# from bpmn_sets import extract_bpmn_sets

# sets1 = extract_bpmn_sets(json.loads(misc_bft_json))
# sets2 = extract_bpmn_sets(model2_aligned)

In [29]:
# second_minimal_json =

new_json = {
    "activities": [
        # changed case, removed whitespace, typo
        {"id": "sid-E5689480-97F3-457D-B623-EF8AA5626B99", "name": "plan travel", "type": "Task"},  # was "Plan travels"
        {
            "id": "sid-C30095C3-EE47-46D1-992F-3D1CBAEE726E",
            "name": "select best offer & request ticket",  # was "Select the best offer and request tickets"
            "type": "Send",
        },
        {
            "id": "sid-BDBF2D9E-F414-4EB1-99E0-1E9C4D319DFF",
            "name": "create schedule",  # was "Create schedule"
            "type": "Task",
        },
        {
            "id": "sid-BD21A3DC-0E6D-46A2-A745-5F69C3991B8E",
            "name": "",  # unchanged, still empty
            "type": "Subprocess",
            "elemRefs": ["sid-044B6E47-536D-4262-B6A0-102D3C20A06A", "sid-619F6299-172A-4AFD-8C60-15267AF1D0C8"],
        },
    ],
    "events": [
        # changed wording/case
        {
            "id": "sid-9D6D682F-6E36-4DDD-BED8-15A5A5EB3296",
            "name": "Wanderlust",  # was "Feeling the Wanderlust"
            "type": "StartNoneEvent",
        },
        {
            "id": "sid-6DE31EA7-8E36-459E-80CC-4E9B21E358B9",
            "name": "send travel req",  # was "Send travel request"
            "type": "IntermediateMessageEventThrowing",
        },
        {
            "id": "sid-C3683C28-0D9F-4D82-A8BA-EC504988B902",
            "name": "schedule received",  # was "Get schedule"
            "type": "IntermediateMessageEventCatching",
        },
        {
            "id": "sid-08598060-42EA-4673-B1AC-33F41067FDC5",
            "name": "Receive e-ticket",  # was "Receive eTickets"
            "type": "IntermediateMessageEventCatching",
        },
        {
            "id": "sid-7DBABD5C-7A71-4530-89FD-EB705ABDB000",
            "name": "Trip can start",  # was "Travel can begin"
            "type": "EndNoneEvent",
        },
        {
            "id": "sid-4F7BA55F-A01B-4E6B-B083-49DDC83AB4F6",
            "name": "Confirm received",  # was "Receive confirmation"
            "type": "IntermediateMessageEventCatching",
        },
        {
            "id": "sid-223B9831-71E3-42E7-B465-894993022280",
            "name": "request arrived",  # was "travel request received"
            "type": "StartMessageEvent",
        },
        {
            "id": "sid-D577E480-5715-4431-9E35-BBDA8806D8F6",
            "name": "Request processed",  # was "Customer request processed"
            "type": "EndMessageEvent",
        },
        {
            "id": "sid-65AC7D4E-E93A-41DF-942C-28A7D96D6275",
            "name": "two min",  # was "2 min"
            "type": "IntermediateTimerEvent",
        },
    ],
    "gateways": [
        # one type intentionally changed to test mapping
        {"id": "sid-8FFC69C7-70EB-4049-83F3-675194B11905", "type": "parallel"},  # lower case, was "Parallel"
        {"id": "sid-CD629142-A4C4-40EE-B7FF-EB0660A3ECAE", "type": "Parallel"},  # unchanged
        {"id": "sid-4780FC45-2589-47CD-8E26-43E64C2D486D", "type": "Exclusive"},  # unchanged
    ],
    "pools": [
        {"id": "sid-847DA9B5-F33A-4DEB-BAF7-06B206AD502E", "name": "Air lines", "lanes": []},  # was "Airline"
        {
            "id": "sid-2F5E65CA-8E92-447B-877E-76FA9C17BBA8",
            "name": "Client",  # was "Customer"
            "lanes": [
                {
                    "id": "sid-3A35FBCA-D99D-41E4-AE73-90E9C6C9AC4A",
                    "name": "",
                    "elemRefs": [
                        "sid-E5689480-97F3-457D-B623-EF8AA5626B99",
                        "sid-9D6D682F-6E36-4DDD-BED8-15A5A5EB3296",
                        "sid-6DE31EA7-8E36-459E-80CC-4E9B21E358B9",
                        "sid-C3683C28-0D9F-4D82-A8BA-EC504988B902",
                        "sid-08598060-42EA-4673-B1AC-33F41067FDC5",
                        "sid-7DBABD5C-7A71-4530-89FD-EB705ABDB000",
                        "sid-8FFC69C7-70EB-4049-83F3-675194B11905",
                        "sid-CD629142-A4C4-40EE-B7FF-EB0660A3ECAE",
                    ],
                }
            ],
        },
        {
            "id": "sid-611624F7-0B93-4DAB-B829-6C66B1CC0181",
            "name": "TravelAgent",  # was "Travel agency"
            "lanes": [
                {
                    "id": "sid-85C8F041-EC3B-451E-BFF9-53FED2FE4EB1",
                    "name": "",
                    "elemRefs": [
                        "sid-C30095C3-EE47-46D1-992F-3D1CBAEE726E",
                        "sid-BDBF2D9E-F414-4EB1-99E0-1E9C4D319DFF",
                        "sid-BD21A3DC-0E6D-46A2-A745-5F69C3991B8E",
                        "sid-4F7BA55F-A01B-4E6B-B083-49DDC83AB4F6",
                        "sid-223B9831-71E3-42E7-B465-894993022280",
                        "sid-D577E480-5715-4431-9E35-BBDA8806D8F6",
                        "sid-65AC7D4E-E93A-41DF-942C-28A7D96D6275",
                        "sid-4780FC45-2589-47CD-8E26-43E64C2D486D",
                    ],
                }
            ],
        },
    ],
    "messageFlows": [
        # unchanged
        {
            "id": "sid-38BE7F50-8A2C-49F4-91C2-5B3BAB34D1A1",
            "targetRef": "sid-223B9831-71E3-42E7-B465-894993022280",
            "sourceRef": "sid-6DE31EA7-8E36-459E-80CC-4E9B21E358B9",
        },
        {"id": "sid-E8B8C182-4982-461D-B47A-DF0989EDF385", "targetRef": "sid-847DA9B5-F33A-4DEB-BAF7-06B206AD502E"},
        {
            "id": "sid-8C2E8C09-94E9-4E5A-AE15-0D694C281ADE",
            "targetRef": "sid-619F6299-172A-4AFD-8C60-15267AF1D0C8",
            "sourceRef": "sid-847DA9B5-F33A-4DEB-BAF7-06B206AD502E",
        },
        {
            "id": "sid-656A3861-9BAE-474B-938D-FAD71080CF81",
            "targetRef": "sid-847DA9B5-F33A-4DEB-BAF7-06B206AD502E",
            "sourceRef": "sid-C30095C3-EE47-46D1-992F-3D1CBAEE726E",
        },
        {
            "id": "sid-E037A05A-6A98-4ED4-8872-A811313B1E99",
            "targetRef": "sid-4F7BA55F-A01B-4E6B-B083-49DDC83AB4F6",
            "sourceRef": "sid-847DA9B5-F33A-4DEB-BAF7-06B206AD502E",
        },
        {
            "id": "sid-55C8EE95-7DDA-479C-AFB5-2CA6C60ADE30",
            "targetRef": "sid-C3683C28-0D9F-4D82-A8BA-EC504988B902",
            "sourceRef": "sid-D577E480-5715-4431-9E35-BBDA8806D8F6",
        },
        {
            "id": "sid-BF340001-59FA-41AA-8EB1-037A8B83BBE5",
            "targetRef": "sid-08598060-42EA-4673-B1AC-33F41067FDC5",
            "sourceRef": "sid-847DA9B5-F33A-4DEB-BAF7-06B206AD502E",
        },
    ],
    "sequenceFlows": [
        # unchanged
        {
            "id": "sid-B05C99F6-88AE-481F-866E-B477C4F03114",
            "targetRef": "sid-E5689480-97F3-457D-B623-EF8AA5626B99",
            "sourceRef": "sid-9D6D682F-6E36-4DDD-BED8-15A5A5EB3296",
        },
        {
            "id": "sid-6CFCAB74-F99D-40F4-8D63-C630067F770C",
            "targetRef": "sid-6DE31EA7-8E36-459E-80CC-4E9B21E358B9",
            "sourceRef": "sid-E5689480-97F3-457D-B623-EF8AA5626B99",
        },
        {
            "id": "sid-29CC4EF1-285F-4448-9F16-9A59BC4D2961",
            "targetRef": "sid-4F7BA55F-A01B-4E6B-B083-49DDC83AB4F6",
            "sourceRef": "sid-C30095C3-EE47-46D1-992F-3D1CBAEE726E",
        },
        {
            "id": "sid-90D1A4A6-96E9-4751-A514-3B89A25D029B",
            "targetRef": "sid-BDBF2D9E-F414-4EB1-99E0-1E9C4D319DFF",
            "sourceRef": "sid-4F7BA55F-A01B-4E6B-B083-49DDC83AB4F6",
        },
        {
            "id": "sid-5519292D-41B6-42EC-A1C1-59301B17A9B1",
            "targetRef": "sid-8FFC69C7-70EB-4049-83F3-675194B11905",
            "sourceRef": "sid-6DE31EA7-8E36-459E-80CC-4E9B21E358B9",
        },
        {
            "id": "sid-FCDE867E-546E-4DB5-A41B-CA0793B8185C",
            "targetRef": "sid-C3683C28-0D9F-4D82-A8BA-EC504988B902",
            "sourceRef": "sid-8FFC69C7-70EB-4049-83F3-675194B11905",
        },
        {
            "id": "sid-1937D7C5-7A29-41CF-A357-01920694D23E",
            "targetRef": "sid-CD629142-A4C4-40EE-B7FF-EB0660A3ECAE",
            "sourceRef": "sid-08598060-42EA-4673-B1AC-33F41067FDC5",
        },
        {
            "id": "sid-057B0310-5D3D-43D5-AC58-401CCB61DFED",
            "targetRef": "sid-CD629142-A4C4-40EE-B7FF-EB0660A3ECAE",
            "sourceRef": "sid-C3683C28-0D9F-4D82-A8BA-EC504988B902",
        },
        {
            "id": "sid-EAF6B6A4-6C14-4BEE-8E3A-13C23337DA96",
            "targetRef": "sid-7DBABD5C-7A71-4530-89FD-EB705ABDB000",
            "sourceRef": "sid-CD629142-A4C4-40EE-B7FF-EB0660A3ECAE",
        },
        {
            "id": "sid-82D08B4D-EAFA-4493-8AEF-B033CA5330B4",
            "targetRef": "sid-D577E480-5715-4431-9E35-BBDA8806D8F6",
            "sourceRef": "sid-BDBF2D9E-F414-4EB1-99E0-1E9C4D319DFF",
        },
        {
            "id": "sid-B40447E2-9A8B-4C23-A2FA-04AC1E0F9F53",
            "targetRef": "sid-BD21A3DC-0E6D-46A2-A745-5F69C3991B8E",
            "sourceRef": "sid-223B9831-71E3-42E7-B465-894993022280",
        },
        {
            "id": "sid-A90FEC34-12FF-4446-B1CA-4296F231B6BC",
            "targetRef": "sid-4780FC45-2589-47CD-8E26-43E64C2D486D",
            "sourceRef": "sid-BD21A3DC-0E6D-46A2-A745-5F69C3991B8E",
        },
        {
            "id": "sid-E1C765D8-E174-417F-AE70-FB6B2336525F",
            "targetRef": "sid-4780FC45-2589-47CD-8E26-43E64C2D486D",
            "sourceRef": "sid-65AC7D4E-E93A-41DF-942C-28A7D96D6275",
        },
        {
            "id": "sid-905145C0-C476-42EF-BACC-F74191B6B437",
            "targetRef": "sid-C30095C3-EE47-46D1-992F-3D1CBAEE726E",
            "sourceRef": "sid-4780FC45-2589-47CD-8E26-43E64C2D486D",
        },
        {
            "id": "sid-8E8E9B6A-B3E5-4218-8F31-CA86D61FD7C5",
            "targetRef": "sid-08598060-42EA-4673-B1AC-33F41067FDC5",
            "sourceRef": "sid-8FFC69C7-70EB-4049-83F3-675194B11905",
        },
        {"id": "sid-8ECDD5DB-9989-4B74-A3BD-1C47BC576CDA", "targetRef": "sid-619F6299-172A-4AFD-8C60-15267AF1D0C8"},
    ],
}

In [58]:
create_all_atomic_mappings(json.loads(misc_bft_json), new_json, bert_cosine_optimized, threshold=0.98)

{'activity_names': {'create schedule': 'Create schedule',
  'plan travel': 'Plan travels'},
 'activity_types': {'Send': 'Send',
  'Subprocess': 'Subprocess',
  'Task': 'Task'},
 'event_types': {'IntermediateTimerEvent': 'IntermediateTimerEvent',
  'StartNoneEvent': 'StartNoneEvent',
  'IntermediateMessageEventThrowing': 'IntermediateMessageEventThrowing',
  'IntermediateMessageEventCatching': 'IntermediateMessageEventCatching',
  'EndNoneEvent': 'EndNoneEvent',
  'EndMessageEvent': 'EndMessageEvent',
  'StartMessageEvent': 'StartMessageEvent'},
 'gateway_types': {'Exclusive': 'Exclusive',
  'Parallel': 'Parallel',
  'parallel': 'Parallel'}}

In [59]:
model2_aligned, mappings = normalize_atomic_names(
    json.loads(misc_bft_json), new_json, bert_cosine_optimized, threshold=0.98
)
# # # 2. Extract sets from both:
# from bpmn_sets import extract_bpmn_sets

# sets1 = extract_bpmn_sets(json.loads(misc_bft_json))
# sets2 = extract_bpmn_sets(model2_aligned)

In [60]:
model2_aligned

{'activities': [{'id': 'sid-E5689480-97F3-457D-B623-EF8AA5626B99',
   'name': 'Plan travels',
   'type': 'Task'},
  {'id': 'sid-C30095C3-EE47-46D1-992F-3D1CBAEE726E',
   'name': 'select best offer & request ticket',
   'type': 'Send'},
  {'id': 'sid-BDBF2D9E-F414-4EB1-99E0-1E9C4D319DFF',
   'name': 'Create schedule',
   'type': 'Task'},
  {'id': 'sid-BD21A3DC-0E6D-46A2-A745-5F69C3991B8E',
   'name': '',
   'type': 'Subprocess',
   'elemRefs': ['sid-044B6E47-536D-4262-B6A0-102D3C20A06A',
    'sid-619F6299-172A-4AFD-8C60-15267AF1D0C8']}],
 'events': [{'id': 'sid-9D6D682F-6E36-4DDD-BED8-15A5A5EB3296',
   'name': 'Wanderlust',
   'type': 'StartNoneEvent'},
  {'id': 'sid-6DE31EA7-8E36-459E-80CC-4E9B21E358B9',
   'name': 'send travel req',
   'type': 'IntermediateMessageEventThrowing'},
  {'id': 'sid-C3683C28-0D9F-4D82-A8BA-EC504988B902',
   'name': 'schedule received',
   'type': 'IntermediateMessageEventCatching'},
  {'id': 'sid-08598060-42EA-4673-B1AC-33F41067FDC5',
   'name': 'Receive e-

In [9]:
import bpmn_similarity
import bpmn_sets

# extract the sets of elements from the minimal json format
bpmn_similarity.extract_bpmn_sets(json.loads(misc_bft_json_original))

{'task_names': ['Enter topic, salary expectations, experience and hobbies',
  'Present offers',
  'Invalidate other offers',
  'Receive status update about internship',
  'Write status update about trainee',
  'Propose recommendation of company to friends of user',
  'Let user decide on friends',
  'Receive list of friends',
  'Create post for each selected friend',
  'Let user approve the posts',
  'Send posts',
  'Ask for reasons',
  'Receive reason'],
 'task_types': ['User',
  'Service',
  'Service',
  'User',
  'Manual',
  'Send',
  'Service',
  'Receive',
  'Script',
  'User',
  'Service',
  'Service',
  'Receive'],
 'event_names': [],
 'event_types': ['StartNoneEvent', 'EndNoneEvent'],
 'gateway_names': ['User decides on offers',
  'Does the user recommend  the company?',
  'User approves?'],
 'gateway_types': ['Exclusive',
  'Parallel',
  'Parallel',
  'Exclusive',
  'Exclusive',
  'Exclusive',
  'Complex'],
 'seq_flows_str': ['Present offers  Exclusive',
  'Enter topic, salary 

In [11]:
bpmn_sets.extract_bpmn_sets(json.loads(misc_bft_json))

{'activity_names': ['Enter topic, salary expectations, experience and hobbies',
  'Present offers',
  'Invalidate other offers',
  'Receive status update about internship',
  'Write status update about trainee',
  'Propose recommendation of company to friends of user',
  'Let user decide on friends',
  'Receive list of friends',
  'Create post for each selected friend',
  'Let user approve the posts',
  'Send posts',
  'Ask for reasons',
  'Receive reason'],
 'activity_types': ['User',
  'Service',
  'Service',
  'User',
  'Manual',
  'Send',
  'Service',
  'Receive',
  'Script',
  'User',
  'Service',
  'Service',
  'Receive'],
 'event_names': [],
 'event_types': ['StartNoneEvent', 'EndNoneEvent'],
 'gateway_names': ['User decides on offers',
  'Does the user recommend  the company?',
  'User approves?'],
 'gateway_types': ['Exclusive',
  'Parallel',
  'Parallel',
  'Exclusive',
  'Exclusive',
  'Exclusive',
  'Complex'],
 'seq_flows_str': ['Present offers  Exclusive',
  'Enter topic,

In [ ]:
bpmn_similarity.calculate_similarity_scores(
    json.loads(pc_json), json.loads(E4_json), method="jaccard", similarity_threshold=0.98
)

In [ ]:
# calculate a similarity score between two minimal jsons
bpmn_similarity.calculate_similarity_alternative(
    json.loads(pc_json), json.loads(E4_json), method="f1", similarity_threshold=0.98
)